# InceptionV1 (GoogLeNet) DPU Benchmark
Measures inference speed and power on the KV260 **FPGA DPU (B512)**.

**Run from Jupyter** (`http://192.168.68.60:9090/lab`)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP, N_BENCHMARK = 10, 100

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL = (os.path.join(NOTEBOOK_DIR, "models", "dpu_tf_inceptionv1.xmodel")
          if os.path.exists(os.path.join(NOTEBOOK_DIR, "models", "dpu_tf_inceptionv1.xmodel"))
          else "/root/jupyter_notebooks/pynq-dpu/dpu_tf_inceptionv1.xmodel")

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"xmodel: {XMODEL}")
print(f"Idle power: {read_power_mw()/1000:.2f} W")

## Step 1 — Load DPU Overlay

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 2 — Load InceptionV1 Model
Output: 1001 classes (index 0 = background, 1-1000 = ImageNet).

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner
in_t, out_t = dpu.get_input_tensors(), dpu.get_output_tensors()
print(f"Input:  {in_t[0].dims}")
print(f"Output: {out_t[0].dims}  (1001: background + 1000 ImageNet classes)")

## Step 3 — Benchmark (zeros input)

In [ ]:
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]

for _ in range(N_WARMUP):
    job = dpu.execute_async(in_d, out_d); dpu.wait(job)
print("Warmup done!")

power_samples, start = [], time.time()
for i in range(N_BENCHMARK):
    job = dpu.execute_async(in_d, out_d); dpu.wait(job)
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start

avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
print("=" * 40)
print("INCEPTIONV1 DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"FPS:      {fps:.1f}")
print(f"Latency:  {elapsed/N_BENCHMARK*1000:.1f} ms/frame")
print(f"Power:    {avg_power_w:.2f} W")
print(f"FPS/Watt: {fps/avg_power_w:.2f}")
print("=" * 40)

In [ ]:
# Release benchmark overlay
del overlay
print("Benchmark overlay released.")

## Bonus — What Does InceptionV1 See? (Real Image Test)
**Preprocessing**: `pixels - 128` — same as ResNet50 (TF model, same Vitis AI compilation).

This section uses its own separate overlay — delete or modify independently of the benchmark above.

In [ ]:
import cv2, json, glob

label_files = [f for f in glob.glob("/home/ubuntu/**/labels.json", recursive=True) +
               ["/home/ubuntu/imagenet_labels.json"] if os.path.exists(f)]
if not label_files:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json",
        "/home/ubuntu/imagenet_labels.json")
    label_files = ["/home/ubuntu/imagenet_labels.json"]
with open(label_files[0]) as f:
    labels = json.load(f)

img_path = next((p for p in [
    "/root/jupyter_notebooks/dpu_benchmark/test_image_224.jpg",
    "/home/ubuntu/test_image_224.jpg"] if os.path.exists(p)), None)
img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# Separate overlay for real image inference
overlay2 = DpuOverlay("dpu.bit")
overlay2.load_model(XMODEL)
dpu2 = overlay2.runner
in_t2, out_t2 = dpu2.get_input_tensors(), dpu2.get_output_tensors()

# pixels-128: confirmed correct for TF models compiled with Vitis AI
img_int8 = (img_rgb.astype(np.int32) - 128).clip(-128, 127).astype(np.int8)

in_d2  = [np.zeros(t.dims, dtype=np.int8) for t in in_t2]
out_d2 = [np.zeros(t.dims, dtype=np.int8) for t in out_t2]
in_d2[0][0] = img_int8
job = dpu2.execute_async(in_d2, out_d2)
dpu2.wait(job)

# Index 0 = background, 1-1000 = ImageNet classes
scores = out_d2[0].flatten().astype(np.float32)
top5 = [int(i) for i in np.argsort(scores)[::-1][:5]]

print(f"Image: {img_path}")
print("InceptionV1 DPU top-5 predictions:")
for r, idx in enumerate(top5):
    label_idx = idx - 1 if idx > 0 else 0
    lbl = labels[label_idx] if label_idx < len(labels) else f"class_{idx}"
    print(f"  {r+1}. {lbl} (score: {scores[idx]:.1f})")

del overlay2
print("\nBonus overlay released.")